# ============================================================
# HYROX BILBAO 2026 — SCRAPING SPLITS POR ESTACIÓN
# Input:  hyrox_bilbao_2026_clean.csv + HTMLs páginas ranking
# Output: hyrox_bilbao_2026_splits.csv

In [1]:
# CELDA 2 — Imports y montar Drive
from google.colab import drive
from bs4 import BeautifulSoup
import pandas as pd
import requests
import time
import os

drive.mount('/content/drive')

RUTA = '/content/drive/MyDrive/hyrox'
print("✅ Drive montado")
print(f"\nArchivos disponibles:")
for a in sorted(os.listdir(RUTA)):
    print(f"  {a}")

Mounted at /content/drive
✅ Drive montado

Archivos disponibles:
  hyox_page0.html
  hyrox_bilbao_2026_clean.csv
  hyrox_bilbao_2026_raw.csv
  hyrox_bilbao_2026_splits.csv
  hyrox_page1.html
  hyrox_page2.html
  hyrox_page3.html
  hyrox_page4.html
  hyrox_page5.html
  hyrox_page6.html
  hyrox_page7.html
  hyrox_page8.html


In [2]:
# CELDA 3 — Extraer URLs de detalle de todos los HTMLs
archivos = ['hyox_page0.html'] + [f'hyrox_page{i}.html' for i in range(1, 9)]

urls_detalle = []
for nombre in archivos:
    with open(f'{RUTA}/{nombre}', 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')

    enlaces = soup.find_all('a', href=lambda h: h and 'content=detail' in h)
    for e in enlaces:
        base = "https://results.hyrox.com/season-8/index.php"
        url = base + e['href']
        nombre_pareja = e.text.strip()
        urls_detalle.append({'nombres_pareja': nombre_pareja, 'url': url})

df_urls = pd.DataFrame(urls_detalle)
df_urls = df_urls.drop_duplicates(subset='url').reset_index(drop=True)

print(f"✅ URLs únicas de detalle: {len(df_urls)}")
print(df_urls.head(3).to_string())

✅ URLs únicas de detalle: 861
                                   nombres_pareja                                                                                                                                                                                                                                                                                             url
0                 Álvaro Villegas, Teresa Bartret  https://results.hyrox.com/season-8/index.php?content=detail&fpid=list&pid=list&idp=LR3MS4JI467233&lang=EN_CAP&event=HD_LR3MS4JI12C3&num_results=100&pidp=ranking_nav&ranking=time_finish_netto&search%5Bsex%5D=M&search%5Bage_class%5D=%25&search%5Bnation%5D=%25&search_event=HD_LR3MS4JI12C3
1  JOSE AGUSTIN ALISES GIMENEZ, LUIS GARCIA RUBIO  https://results.hyrox.com/season-8/index.php?content=detail&fpid=list&pid=list&idp=LR3MS4JI467DDF&lang=EN_CAP&event=HD_LR3MS4JI12C3&num_results=100&pidp=ranking_nav&ranking=time_finish_netto&search%5Bsex%5D=M&search%5Bage_class%5D=%25&search%5

In [3]:
# CELDA 4 — Función de extracción de splits
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"}

def extraer_splits(url):
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, 'html.parser')

    tablas = soup.find_all('table')
    if len(tablas) < 7:
        return None

    tabla_splits = tablas[6]
    filas = tabla_splits.find_all('tr')

    mapa = {
        'f-time_01': 'running_1',
        'f-time_11': '1000m_skierg',
        'f-time_02': 'running_2',
        'f-time_12': '50m_sled_push',
        'f-time_03': 'running_3',
        'f-time_13': '50m_sled_pull',
        'f-time_04': 'running_4',
        'f-time_14': '80m_burpee_broad_jump',
        'f-time_05': 'running_5',
        'f-time_15': '1000m_row',
        'f-time_06': 'running_6',
        'f-time_16': '200m_farmers_carry',
        'f-time_07': 'running_7',
        'f-time_17': '100m_sandbag_lunges',
        'f-time_08': 'running_8',
        'f-time_18': 'wall_balls',
        'f-time_60': 'roxzone_time',
        'f-time_49': 'run_total',
        'f-time_50': 'best_run_lap',
    }

    resultado = {}
    for fila in filas:
        clases = fila.get('class', [])
        for clase, nombre in mapa.items():
            if clase in clases:
                tds = fila.find_all('td')
                resultado[nombre] = tds[0].text.strip() if len(tds) > 0 else None
                break

    return resultado

In [4]:
# CELDA 5 — Test con una URL antes de lanzar el scraper completo
url_test = df_urls.iloc[0]['url']
print(f"Testeando: {df_urls.iloc[0]['nombres_pareja']}")
splits = extraer_splits(url_test)
for k, v in splits.items():
    print(f"  {k}: {v}")

Testeando: Álvaro Villegas, Teresa Bartret
  running_1: 00:04:34
  1000m_skierg: 00:04:05
  running_2: 00:04:28
  50m_sled_push: 00:02:18
  running_3: 00:04:36
  50m_sled_pull: 00:02:52
  running_4: 00:04:40
  80m_burpee_broad_jump: 00:02:42
  running_5: 00:05:06
  1000m_row: 00:04:28
  running_6: 00:04:41
  200m_farmers_carry: 00:01:28
  running_7: 00:04:44
  100m_sandbag_lunges: 00:03:03
  running_8: 00:04:55
  wall_balls: 00:04:06
  roxzone_time: 00:05:05
  run_total: 00:37:41
  best_run_lap: 00:04:28


In [5]:
# CELDA 6 — Scraper completo (aprox. 7-8 minutos)
todos_los_splits = []
errores = []

print(f"Total URLs a procesar: {len(df_urls)}")
print("Iniciando scraping...\n")

for i, row in df_urls.iterrows():
    try:
        splits = extraer_splits(row['url'])
        if splits:
            splits['nombres_pareja'] = row['nombres_pareja']
            todos_los_splits.append(splits)
        else:
            errores.append({'idx': i, 'nombres': row['nombres_pareja'], 'error': 'sin tabla'})
    except Exception as e:
        errores.append({'idx': i, 'nombres': row['nombres_pareja'], 'error': str(e)})

    if (i + 1) % 50 == 0:
        print(f"  ✅ {i + 1}/{len(df_urls)} procesadas...")

    time.sleep(0.5)

df_splits = pd.DataFrame(todos_los_splits)
print(f"\nCompletado:")
print(f"  ✅ Splits extraídos: {len(df_splits)}")
print(f"  ❌ Errores: {len(errores)}")

Total URLs a procesar: 861
Iniciando scraping...



KeyboardInterrupt: 

In [ ]:
# CELDA 7 — Verificar resultado
print("=== SHAPE ===")
print(df_splits.shape)

print("\n=== NULOS POR COLUMNA ===")
print(df_splits.isnull().sum())

print("\n=== MUESTRA ===")
print(df_splits.head(5).to_string())

In [ ]:
# CELDA 8 — Guardar CSV
df_splits.to_csv(f'{RUTA}/hyrox_bilbao_2026_splits.csv', index=False)
print("✅ Guardado: hyrox_bilbao_2026_splits.csv")
print(f"   {len(df_splits)} registros | {df_splits.shape[1]} columnas")